# # Summarizing Reddit Subreddit Posts



## Step 1: Import libraries

In [39]:
import os
from tavily import TavilyClient
from dotenv import load_dotenv
import json
from openai import OpenAI
from utils import function_to_tool
from IPython.display import display, Markdown
import gradio as gr
from datetime import datetime, timedelta, timezone
import requests

load_dotenv()

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY is not set in the environment variables.")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is not set in the environment variables.")

tavily_client = TavilyClient()
openai_client = OpenAI()

You can setup your API key here: **[Tavily API Key](https://app.tavily.com/home)**

## Step 2: Define our tools

**The tools to define:**

1. Flight Search Tool
2. Hotel Search Tool

In [3]:
query = "Summarize the MicrosoftFabric subreddit posts from yesterday."

response = tavily_client.search(
    query=query,
    include_domains=["reddit.com"]
)

print(json.dumps(response, indent=2))

{
  "query": "Summarize the MicrosoftFabric subreddit posts from yesterday.",
  "follow_up_questions": null,
  "answer": null,
  "images": [],
  "results": [
    {
      "url": "https://www.reddit.com/r/dataengineering/comments/1jvwwcy/tried_to_roll_out_microsoft_fabric_ended_up/",
      "title": "Tried to roll out Microsoft Fabric\u2026 ended up rolling straight into a ...",
      "content": "Yesterday morning, all capacity in a Microsoft Fabric production environment was completely drained \u2014 and it's only April. What happened?",
      "score": 0.6519982,
      "raw_content": null
    },
    {
      "url": "https://www.reddit.com/r/MicrosoftFabric/comments/1nib2ov/fabric_september_2025_feature_summary_microsoft/",
      "title": "Fabric September 2025 Feature Summary | Microsoft Fabric Blog",
      "content": "Great great updates! A lot of things to dig into. Also the new tabbed experience is perfect. Also happy to see schema in lakehouse makes it",
      "score": 0.6301622,
    

In [4]:
def subreddit_search(query: str) -> str:
    """
    Searches for posts in a Reddit subreddit based on the provided query

    Args:
        query (str): The search query for subreddit posts

    Returns:
        str: A formatted string containing the search results
    
    """

    response = tavily_client.search(
    query=query,
    include_domains=["reddit.com"]
    )

    results = response.get("results", [])

    # extract items
    contents = [item.get("content", "") for item in results]

    # Format as a numbered list
    formatted_contents = "\n".join(f"{i + 1}. {content}" for i, content in enumerate(contents) if content)

    return formatted_contents

In [5]:
print(subreddit_search(query))

1. I keep seeing multiple posts here and in r/PowerBI asking the same things about PL-300 and DP-600/DP-700 that can be summarized "how did you
2. Anyone that works with data knows one thing - whats important, is reliability. That's it. If something does not work - thats completely fine.
3. New post where I want to encourage others to think about their Microsoft Fabric Continuous Integration maturity levels.
4. The new Fabric community blogs are now available. Which you can access by clicking the link below: Fabric community blogs - Microsoft Fabric Community.
5. Welcome u/aleonard763 !!! love getting more and more of our Data Factory community experts jumping into the forums :).


In [6]:
query = "What is the most commented post in the MicrosoftFabric subreddit from yesterday and what are the comments about?"

print(subreddit_search(query))

## Step 3: Define our tools schema

In [7]:
subreddit_tool_schema = function_to_tool(subreddit_search)

In [8]:
subreddit_tool_schema

{'type': 'function',
 'name': 'subreddit_search',
 'description': 'Searches for posts in a Reddit subreddit based on the provided query',
 'parameters': {'type': 'object',
  'properties': {'query': {'type': 'string',
    'description': 'The search query for subreddit posts'}},
  'required': ['query']}}

## Step 4: Define a Prompt Template

Prompt templates are discussed in our [Prompt Engineering course](https://github.com/SuperDataScience-Community/prompt-engineering) specifically in the notebook for [multi-shot prompting](https://github.com/SuperDataScience-Community/prompt-engineering/blob/main/prompt-engineering-techniques/multi-shot-prompting.ipynb)

In [9]:
# Class to define prompt template
class PromptTemplate:
    def __init__(self, template: str, input_variables: list[str]):
        self.template = template
        self.input_variables = input_variables

    def generate(self, **kwargs) -> str:
        return self.template.format(**{k: kwargs[k] for k in self.input_variables})

In [ ]:
prompt = PromptTemplate(
    template="I want to know about posts in {subreddit} from {date}. the kinds of posts I prefer are {preferences}",
    input_variables=["subreddit", "date", "preferences"]
)

In [13]:
prompt.generate(subreddit="MicrosoftFabric", date="2026-05-18", preferences="most commented")

'I want to know about posts in MicrosoftFabric from 2026-05-18. the kinds of posts I prefer are most commented'

## Step 5: Call the OpenAI Responses API

In [14]:
system_message = """
You are a Reddit subreddit summarization assistant.

The user will ask for a summary of posts from a specific Reddit subreddit, usually for a specific date such as yesterday.

Your job is to:

1. Use the subreddit search tool to retrieve Reddit posts related to the user's request.
2. Summarize only the information returned by the tool.
3. Do not invent posts, comments, links, trends, or opinions that are not present in the tool results.
4. If the tool returns little or no useful information, clearly say that the available results were limited.
5. Keep the summary concise, structured, and easy to read.
6. Focus on the user's stated preferences if they provide any.
7. Do not ask follow-up questions. Use the information given.

Your final response should include:

- A short overall summary
- Main topics discussed
- Notable posts or themes
- Repeated questions, issues, or complaints
- A brief takeaway

Only return the final subreddit summary. Do not explain the tool call process.
Always call the subreddit search tool before answering.
"""

In [15]:
# user prompt

user_prompt = prompt.generate(
    subreddit="MicrosoftFabric",
    date="2026-05-18",
    preferences="most commented"
)

In [16]:
# input list
input_list = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
]

In [18]:
# response

response = openai_client.responses.create(
    model="gpt-4.1-nano",
    input=input_list,
    tools=[subreddit_tool_schema],
    tool_choice="auto",
    parallel_tool_calls=False
)

if response.output[0].type == "message":
    input_list.append({"role": "assistant", "content": response.output_text})
if response.output[0].type == "function_call":
    input_list += response.output


In [19]:
# print response
input_list

[{'role': 'system',
  'content': "\nYou are a Reddit subreddit summarization assistant.\n\nThe user will ask for a summary of posts from a specific Reddit subreddit, usually for a specific date such as yesterday.\n\nYour job is to:\n\n1. Use the subreddit search tool to retrieve Reddit posts related to the user's request.\n2. Summarize only the information returned by the tool.\n3. Do not invent posts, comments, links, trends, or opinions that are not present in the tool results.\n4. If the tool returns little or no useful information, clearly say that the available results were limited.\n5. Keep the summary concise, structured, and easy to read.\n6. Focus on the user's stated preferences if they provide any.\n7. Do not ask follow-up questions. Use the information given.\n\nYour final response should include:\n\n- A short overall summary\n- Main topics discussed\n- Notable posts or themes\n- Repeated questions, issues, or complaints\n- A brief takeaway\n\nOnly return the final subreddi

## Step 6: Handle tools calls

In [20]:
def call_function(name, args):
    if name == "subreddit_search":
        return subreddit_search(**args)
    else:
        raise ValueError(f"Unknown function: {name}")

In [21]:
name = response.output[0].name
args = json.loads(response.output[0].arguments)

result = call_function(name, args)

input_list.append({
    'type': 'function_call_output',
    'call_id': response.output[0].call_id,
    'output': str(result)
})

In [22]:
input_list

[{'role': 'system',
  'content': "\nYou are a Reddit subreddit summarization assistant.\n\nThe user will ask for a summary of posts from a specific Reddit subreddit, usually for a specific date such as yesterday.\n\nYour job is to:\n\n1. Use the subreddit search tool to retrieve Reddit posts related to the user's request.\n2. Summarize only the information returned by the tool.\n3. Do not invent posts, comments, links, trends, or opinions that are not present in the tool results.\n4. If the tool returns little or no useful information, clearly say that the available results were limited.\n5. Keep the summary concise, structured, and easy to read.\n6. Focus on the user's stated preferences if they provide any.\n7. Do not ask follow-up questions. Use the information given.\n\nYour final response should include:\n\n- A short overall summary\n- Main topics discussed\n- Notable posts or themes\n- Repeated questions, issues, or complaints\n- A brief takeaway\n\nOnly return the final subreddi

## Step 7: The App Logic

<div style="border-radius:16px;background:#2e3440;margin:1em 0;padding:1em;color:#eceff4;position:relative;box-shadow:0 6px 16px rgba(0,0,0,.4);overflow-wrap:break-word;word-break:break-word;">
  <b style="color:#88c0d0;font-size:1.25em">Info:</b>
  <span style="display:block;margin-top:.6em;padding-left:1.2em;line-height:1.6">
    In the video lectures, we saw that our Agent, Atlas was not following the example itinerary properly. This is probably due to the smaller sized model we are using. Try replacing that model with <code>gpt-4o</code>, <code>gpt-5</code> or if you want to continue with cheaper models, <code>gpt-4o-mini</code> all of which are great at tool use (with gpt-5 obviously outperforming all models in agentic use cases).
  </span>
  <div style="position:absolute;top:-.8em;left:-.8em;width:2.4em;height:2.4em;border-radius:50%;background:#88c0d0;color:#2e3440;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:1.2em">💡</div>
</div>

In [26]:
subreddit_tool_schema = {
    "type": "function",
    "function": {
        "name": "subreddit_search",
        "description": "Searches for Reddit posts from a specific subreddit and date using Tavily.",
        "parameters": {
            "type": "object",
            "properties": {
                "subreddit": {
                    "type": "string",
                    "description": "The subreddit name without r/, for example MicrosoftFabric"
                },
                "date": {
                    "type": "string",
                    "description": "The date or time period to search for, for example yesterday or 2026-05-18"
                },
                "preferences": {
                    "type": "string",
                    "description": "Optional user preferences for the kinds of posts to focus on"
                }
            },
            "required": ["subreddit", "date", "preferences"],
            "additionalProperties": False
        }
    }
}

In [27]:
def get_response(input_list):
    response = openai_client.responses.create(
        model="gpt-4.1-nano",
        input=input_list,
        tools=[subreddit_tool_schema],
        tool_choice="auto",
        parallel_tool_calls=False
    )
    return response

In [34]:
def subreddit_search(
    subreddit: str,
    date: str = "yesterday",
    limit: int = 100
) -> str:
    """
    Fetches Reddit posts from a specific subreddit and returns posts from the requested date,
    including title, text, score, number of comments, and URL.

    Args:
        subreddit (str): The subreddit name without r/, for example MicrosoftFabric.
        date (str): The date to search for. Supports "yesterday" or YYYY-MM-DD.
        limit (int): Number of latest posts to check.

    Returns:
        str: A formatted string containing matching Reddit posts.
    """

    subreddit = subreddit.replace("r/", "").strip()

    url = f"https://www.reddit.com/r/{subreddit}/new.json?limit={limit}"

    headers = {
        "User-Agent": "agentic-ai-lab-by-tatiana/0.1"
    }

    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()

    data = response.json()

    now = datetime.now(timezone.utc)

    if date.lower().strip() == "yesterday":
        target_start = datetime(
            year=now.year,
            month=now.month,
            day=now.day,
            tzinfo=timezone.utc
        ) - timedelta(days=1)
    else:
        target_start = datetime.strptime(date.strip(), "%Y-%m-%d").replace(
            tzinfo=timezone.utc
        )

    target_end = target_start + timedelta(days=1)

    matching_posts = []

    for item in data.get("data", {}).get("children", []):
        post = item.get("data", {})

        post_time = datetime.fromtimestamp(
            post.get("created_utc", 0),
            tz=timezone.utc
        )

        if not (target_start <= post_time < target_end):
            continue

        matching_posts.append({
            "title": post.get("title", ""),
            "text": post.get("selftext", ""),
            "score": post.get("score", 0),
            "num_comments": post.get("num_comments", 0),
            "url": "https://www.reddit.com" + post.get("permalink", ""),
            "created_utc": post_time.strftime("%Y-%m-%d %H:%M UTC")
        })

    if not matching_posts:
        return f"No posts found in r/{subreddit} for {date}."

    matching_posts = sorted(
        matching_posts,
        key=lambda post: (post["num_comments"], post["score"]),
        reverse=True
    )

    formatted_posts = []

    for i, post in enumerate(matching_posts, start=1):
        formatted_posts.append(
            f"""
Post {i}
Title: {post['title']}
Text: {post['text']}
Score: {post['score']}
Comments: {post['num_comments']}
Created: {post['created_utc']}
URL: {post['url']}
"""
        )

    return "\n\n".join(formatted_posts)

In [40]:
subreddit_search("MicrosoftFabric", "2026-05-18", limit=100)

'\nPost 1\nTitle: Service entirely down\nText: After some slowness all day UK South the service now seems to be completely down. Any news as to when it will be back?\nScore: 51\nComments: 105\nCreated: 2026-05-18 15:19 UTC\nURL: https://www.reddit.com/r/MicrosoftFabric/comments/1tgps74/service_entirely_down/\n\n\n\nPost 2\nTitle: How to code first Batch ELT in Fabric?\nText: Jumping on the back of posts like these:  \n[Materialized Lake Views: It was too good to be true...](https://www.reddit.com/r/MicrosoftFabric/comments/1tdvxtp/materialized_lake_views_it_was_too_good_to_be_true/)  \n[What\'s the preferred tool to use for medallion architecture in Fabric Lakehouse?](https://www.reddit.com/r/MicrosoftFabric/comments/1tfw3ct/may_2026_whats_the_preferred_tool_to_use_for/)  \n[Warehouse workflow, what works?](https://www.reddit.com/r/MicrosoftFabric/comments/1sau6u2/warehouse_workflow_what_works/) (My post)\n\nI think there is still a void in Fabric when it comes to batch ELT using medal

In [41]:
def summarize_subreddit_posts(
    subreddit: str,
    date: str = "yesterday",
    preferences: str = "",
    limit: int = 100
) -> str:
    """
    Retrieves Reddit posts from a subreddit and summarizes them using OpenAI.
    """

    posts = subreddit_search(
        subreddit=subreddit,
        date=date,
        limit=limit
    )

    if posts.startswith("No posts found"):
        return posts

    system_message = """
You are a Reddit subreddit summarization assistant.

Your job is to summarize Reddit posts retrieved from a subreddit.

Rules:
- Summarize only the information provided in the retrieved posts.
- Do not invent posts, comments, links, opinions, or trends.
- If the available posts are limited, mention that clearly.
- Keep the summary concise, structured, and easy to read.
- Use the user's preferences only to decide what to emphasize in the summary.
- Give more attention to posts with higher comment counts and higher scores.
"""

    user_message = f"""
Subreddit: r/{subreddit}
Date: {date}
User preferences for the summary: {preferences}

Retrieved posts:
{posts}

Please provide:
1. A short overall summary
2. Main topics discussed
3. Most discussed posts
4. Posts with notable scores
5. Repeated issues, questions, or themes
6. A brief takeaway

When relevant, emphasize the user's preferences:
{preferences}

Be concise and clear.
Only return the final subreddit summary. Do not explain the tool call process.
Always call the subreddit search tool before answering.
"""

    openai_response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ]
    )

    return openai_response.choices[0].message.content

In [42]:
display(Markdown(summarize_subreddit_posts("MicrosoftFabric", "2026-05-18", "more comments")))

### Subreddit Summary: r/MicrosoftFabric (May 18, 2026)

1. **Overall Summary**: The subreddit is currently active with discussions revolving around service outages, coding practices in Microsoft Fabric, exam preparations, deployment issues, and authentication challenges. The most engagement is seen with posts related to service disruptions and batch ELT coding.

2. **Main Topics Discussed**:
   - Service outages and performance issues in UK South.
   - Code-first practices for batch ELT in Microsoft Fabric.
   - Guidance and tips for the DP-700 certification exam.
   - Authentication issues with Azure Analysis Services.
   - Deployment problems with notebook resources via Git.

3. **Most Discussed Posts**:
   - **Service entirely down**: 105 comments discussing the complete service outage in UK South and inquiries about restoration timelines. 
   - **How to code first Batch ELT in Fabric?**: 8 comments focused on the challenges and experiences of implementing batch ELT in Fabric.
   - **Azure Analysis Services on Fabric Lakehouse**: 4 comments seeking solutions for authentication issues.

4. **Posts with Notable Scores**:
   - **Service entirely down**: Score of 49.
   - **How to code first Batch ELT in Fabric?**: Score of 8.
   - Posts about the DP-700 exam and deployment issues had scores of 2.

5. **Repeated Issues, Questions, or Themes**:
   - Ongoing service disruptions and CORS errors impacting refresh schedules.
   - Challenges with Git integration for notebook resources.
   - Requests for guidance on exam preparations and usage of features within Microsoft Fabric.

6. **Takeaway**: Users are mainly concerned with service reliability and effective practices in utilizing Microsoft Fabric, highlighting a significant need for support and community engagement during outages and technical challenges.

## Step 8: Gradio UI

In [43]:
custom_css = """
:root {
    --reddit-orange: #ff4500;
    --reddit-orange-dark: #d93a00;
    --reddit-bg: #fff7f3;
    --reddit-card: #ffffff;
    --reddit-text: #1c1c1c;
    --reddit-muted: #6b7280;
}

.gradio-container {
    background: linear-gradient(135deg, #fff7f3 0%, #fff1eb 45%, #ffffff 100%) !important;
    font-family: Inter, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
}

#app-container {
    max-width: 980px;
    margin: 0 auto;
}

#hero {
    background: linear-gradient(135deg, #ff4500 0%, #ff7a1a 100%);
    color: white;
    padding: 32px;
    border-radius: 24px;
    box-shadow: 0 18px 40px rgba(255, 69, 0, 0.22);
    margin-bottom: 24px;
}

#hero h1 {
    font-size: 38px;
    margin-bottom: 8px;
}

#hero p {
    font-size: 17px;
    opacity: 0.95;
    margin-bottom: 0;
}

.input-card {
    background: white;
    border-radius: 22px;
    padding: 22px;
    box-shadow: 0 12px 30px rgba(17, 24, 39, 0.08);
    border: 1px solid rgba(255, 69, 0, 0.12);
}

.output-card {
    background: white;
    border-radius: 22px;
    padding: 22px;
    box-shadow: 0 12px 30px rgba(17, 24, 39, 0.08);
    border: 1px solid rgba(255, 69, 0, 0.12);
}

#submit-btn {
    background: linear-gradient(135deg, #ff4500 0%, #ff7a1a 100%) !important;
    color: white !important;
    border: none !important;
    border-radius: 16px !important;
    font-weight: 700 !important;
    font-size: 16px !important;
    padding: 12px 18px !important;
    box-shadow: 0 10px 22px rgba(255, 69, 0, 0.28) !important;
}

#submit-btn:hover {
    background: linear-gradient(135deg, #d93a00 0%, #ff6500 100%) !important;
    transform: translateY(-1px);
}

#clear-btn {
    border-radius: 16px !important;
}

.gr-textbox textarea,
.gr-textbox input {
    border-radius: 14px !important;
}

.gr-slider {
    border-radius: 14px !important;
}

#tips {
    background: #fff1eb;
    border-left: 5px solid #ff4500;
    padding: 14px 18px;
    border-radius: 16px;
    color: #3a1d12;
    margin-top: 12px;
}

footer {
    visibility: hidden;
}
"""

In [44]:
def gradio_summarize_subreddit(subreddit, date, preferences, limit):
    try:
        if not subreddit or not subreddit.strip():
            return "Please enter a subreddit name, for example `MicrosoftFabric`."

        if not date or not date.strip():
            date = "yesterday"

        return summarize_subreddit_posts(
            subreddit=subreddit,
            date=date,
            preferences=preferences,
            limit=int(limit)
        )

    except ValueError as e:
        return f"""
### Date format issue

Please use either:

- `yesterday`
- `YYYY-MM-DD`, for example `2026-05-18`

Error details: `{str(e)}`
"""

    except Exception as e:
        return f"""
### Something went wrong

Error details:

`{str(e)}`
"""

In [45]:
with gr.Blocks(
    css=custom_css,
    title="Reddit Subreddit Summarizer",
    theme=gr.themes.Soft(
        primary_hue="orange",
        secondary_hue="red",
        neutral_hue="slate"
    )
) as demo:

    with gr.Column(elem_id="app-container"):

        gr.HTML(
            """
            <div id="hero">
                <h1>🔥 Reddit Subreddit Summarizer</h1>
                <p>
                    Pick a subreddit, choose a date, and get a concise AI-powered summary of the most discussed posts.
                </p>
            </div>
            """
        )

        with gr.Row():

            with gr.Column(scale=1, elem_classes="input-card"):
                gr.Markdown("## Search settings")

                subreddit_input = gr.Textbox(
                    label="Subreddit",
                    value="MicrosoftFabric",
                    placeholder="Example: MicrosoftFabric, PowerBI, datascience",
                    info="Enter the subreddit name without r/"
                )

                date_input = gr.Textbox(
                    label="Date",
                    value="yesterday",
                    placeholder="yesterday or 2026-05-18",
                    info="Use 'yesterday' or a date in YYYY-MM-DD format"
                )

                preferences_input = gr.Textbox(
                    label="Summary preferences",
                    value="Focus on Power BI, Fabric, semantic models, data engineering, common issues, and highly discussed posts.",
                    placeholder="Example: focus on technical issues, questions, complaints, tutorials, or highly discussed posts",
                    lines=5,
                    info="These preferences are used only by OpenAI when creating the summary"
                )

                limit_input = gr.Slider(
                    label="Number of latest posts to check",
                    minimum=10,
                    maximum=500,
                    value=100,
                    step=10,
                    info="Higher values check more posts but may take longer"
                )

                with gr.Row():
                    submit_btn = gr.Button(
                        "Summarize Subreddit 🚀",
                        elem_id="submit-btn",
                        scale=2
                    )

                    clear_btn = gr.ClearButton(
                        components=[
                            subreddit_input,
                            date_input,
                            preferences_input
                        ],
                        value="Clear",
                        elem_id="clear-btn",
                        scale=1
                    )

                gr.HTML(
                    """
                    <div id="tips">
                        <strong>Tip:</strong> If the subreddit is very active, increase the post limit to 300–500.
                        If it is quiet, 100 is usually enough.
                    </div>
                    """
                )

            with gr.Column(scale=2, elem_classes="output-card"):
                gr.Markdown("## Summary")

                output = gr.Markdown(
                    value="Your subreddit summary will appear here.",
                    label="Subreddit Summary"
                )

        gr.Examples(
            examples=[
                [
                    "MicrosoftFabric",
                    "yesterday",
                    "Focus on Power BI, Fabric, semantic models, data engineering, common issues, and highly discussed posts.",
                    100
                ],
                [
                    "PowerBI",
                    "yesterday",
                    "Focus on user problems, dashboard performance, DAX, semantic models, and practical tips.",
                    200
                ],
                [
                    "datascience",
                    "yesterday",
                    "Focus on career advice, project ideas, machine learning, and beginner questions.",
                    200
                ]
            ],
            inputs=[
                subreddit_input,
                date_input,
                preferences_input,
                limit_input
            ],
            label="Try an example"
        )

        submit_btn.click(
            fn=gradio_summarize_subreddit,
            inputs=[
                subreddit_input,
                date_input,
                preferences_input,
                limit_input
            ],
            outputs=output,
            show_progress="full"
        )

demo.queue()
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Step 9: Deployment

You should first get setup on huggingface by making an account:

1. Visit the [huggingface](https://huggingface.co/) website and create an account.
2. Create an [API Key](https://huggingface.co/settings/tokens)
3. Copy the app logic into a `app.py` file and include a `requirements.txt` file inside of `community-contributions/your-name`
4. Open a new terminal
5. Activate our virtual environment
    - Windows command: `.venv\Scripts\activate`
    - Mac/Linux command: `source .venv/bin/activate`
6. cd into part1-fundementals using the command `cd part1-fundementals`
7. Run the command `gradio deploy`
8. Open the link to your deployed app

<div style="border-radius:16px;background:#3b1c1c;margin:1em 0;padding:1em;color:#eceff4;position:relative;box-shadow:0 6px 16px rgba(0,0,0,.4);overflow-wrap:break-word;word-break:break-word;">
  <b style="color:#bf616a;font-size:1.25em">Warning:</b>
  <span style="display:block;margin-top:.6em;padding-left:1.2em;line-height:1.6">
    You must add your <code>app.py</code>, <code>requirements.txt</code>, and a copy of <code>utils.py</code> inside the <code>community-contributions/your-name</code> folder.<br>
    For example: <code>community-contributions/your-name/app.py</code><br>
    Replace <code>your-name</code> with your actual name.
  </span>
  <div style="position:absolute;top:-.8em;left:-.8em;width:2.4em;height:2.4em;border-radius:50%;background:#bf616a;color:#2e3440;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:1.2em">⚠️</div>
</div>